# Homework 3 — Improved Retrieval Pipeline

This notebook improves the semantic retrieval pipeline created in Homework 2.

The same knowledge base, processed chunks, embedding model, FAISS index, and test queries are reused so that the baseline and improved retrieval results can be compared fairly.

The experiment includes:

1. baseline retrieval diagnostics;
2. metadata filtering;
3. simple hybrid retrieval;
4. baseline vs improved evaluation;
5. generation of the final homework artifacts.

## Stage 1 — Connect the repository and prepare the environment


In [1]:
!pip install -q sentence-transformers faiss-cpu pandas tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 46.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess


GITHUB_USER = "swanksenia"
REPO_NAME = "health-psychology-rag-kb"
BRANCH = "main"

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
PROJECT_ROOT = Path("/content") / REPO_NAME


github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )


# Temporary helper for secure GitHub authentication.
askpass_path = Path("/content/git_askpass.sh")

askpass_path.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
)

askpass_path.chmod(0o700)


git_environment = os.environ.copy()
git_environment["GITHUB_TOKEN"] = github_token
git_environment["GIT_ASKPASS"] = str(askpass_path)
git_environment["GIT_TERMINAL_PROMPT"] = "0"


try:
    if (PROJECT_ROOT / ".git").exists():
        command = [
            "git",
            "-C",
            str(PROJECT_ROOT),
            "pull",
            "--ff-only",
            "origin",
            BRANCH,
        ]
        action = "updated"

    else:
        command = [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            REPO_URL,
            str(PROJECT_ROOT),
        ]
        action = "cloned"

    subprocess.run(
        command,
        env=git_environment,
        check=True,
        capture_output=True,
        text=True,
    )

    os.chdir(PROJECT_ROOT)

    print(f"✅ Repository successfully {action}.")
    print(f"Project root: {PROJECT_ROOT}")
    print(f"Current directory: {Path.cwd()}")

except subprocess.CalledProcessError as error:
    error_message = error.stderr or "Unknown Git error"
    error_message = error_message.replace(github_token, "***")

    raise RuntimeError(
        f"GitHub connection failed:\n{error_message}"
    ) from error

finally:
    askpass_path.unlink(missing_ok=True)

    git_environment.pop("GITHUB_TOKEN", None)
    git_environment.pop("GIT_ASKPASS", None)

    del github_token
    del git_environment

✅ Repository successfully cloned.
Project root: /content/health-psychology-rag-kb
Current directory: /content/health-psychology-rag-kb


## Stage 2 — Inspect the Homework 2 retrieval artifacts

We inspect the chunk structure and available metadata fields before adding metadata filtering.

We also confirm that the number of vectors in the FAISS index matches the number of retrieval chunks.

In [3]:
import json
import faiss


CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_for_retrieval.jsonl"
)

INDEX_PATH = (
    PROJECT_ROOT
    / "index"
    / "faiss.index"
)

print("Chunks path:", CHUNKS_PATH)
print("Index path:", INDEX_PATH)

Chunks path: /content/health-psychology-rag-kb/data/processed/chunks_for_retrieval.jsonl
Index path: /content/health-psychology-rag-kb/index/faiss.index


In [4]:
def load_jsonl(file_path):
    records = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))

    return records


chunks = load_jsonl(CHUNKS_PATH)

print(f"Loaded chunks: {len(chunks)}")

Loaded chunks: 474


In [5]:
print(
    json.dumps(
        chunks[0],
        indent=2,
        ensure_ascii=False,
    )
)

{
  "chunk_id": "health_psychology_course_syllabus__0000",
  "document_id": "health_psychology_course_syllabus",
  "source_file": "data/raw/course_syllabus.pdf",
  "chunk_index": 0,
  "section": "Document overview",
  "text": "Syllabus for Introduction to Health Psychology Credits: 3 PSYC 1111 Instructor Contact Information: You can always send your instructor a private message through the Brightspace Messaging system, accessible via the envelope (Messages) icon in the top navigation bar. Once logged into your course, click your instructor’s profile page to see all the ways you can communicate with them, including their email address. Course Description Health psychology focuses on the dynamic interaction between biological, social, and psychological factors that influence physical health and illness, aiming to promote overall well-being and prevent diseases. This course is designed to provide students with an introduction to the field of health psychology."
}


In [7]:
metadata_fields = [
    "document_id",
    "source_file",
    "section",
    "chunk_index",
]

print("Available metadata fields:")

for field in metadata_fields:
    if field in chunks[0]:
        print("-", field)

Available metadata fields:
- document_id
- source_file
- section
- chunk_index


In [9]:
for chunk in chunks:
    chunk["metadata"] = {
        "document_id": chunk.get("document_id"),
        "source_file": chunk.get("source_file"),
        "section": chunk.get("section"),
        "chunk_index": chunk.get("chunk_index"),
    }

print(
    json.dumps(
        chunks[0]["metadata"],
        indent=2,
        ensure_ascii=False,
    )
)

{
  "document_id": "health_psychology_course_syllabus",
  "source_file": "data/raw/course_syllabus.pdf",
  "section": "Document overview",
  "chunk_index": 0
}


In [10]:
index = faiss.read_index(str(INDEX_PATH))

print("Indexed vectors:", index.ntotal)
print("Retrieval chunks:", len(chunks))

if index.ntotal != len(chunks):
    raise ValueError(
        "The number of FAISS vectors does not match "
        "the number of retrieval chunks."
    )

print("✅ FAISS index matches the chunk collection.")

Indexed vectors: 474
Retrieval chunks: 474
✅ FAISS index matches the chunk collection.


In [11]:
from collections import Counter


document_counts = Counter(
    chunk["document_id"]
    for chunk in chunks
)

print("Documents in the retrieval dataset:\n")

for document_id, chunk_count in sorted(document_counts.items()):
    print(f"- {document_id}: {chunk_count} chunks")

Documents in the retrieval dataset:

- health_psychology_course_syllabus: 23 chunks
- michie_2011_behaviour_change_wheel: 82 chunks
- ogden_2019_health_psychology: 247 chunks
- wright_2019_3p_disease_model: 122 chunks


In [12]:
document_sources = {}

for chunk in chunks:
    document_sources[
        chunk["document_id"]
    ] = chunk["source_file"]


print("Document sources:\n")

for document_id, source_file in sorted(
    document_sources.items()
):
    print(f"- {document_id}")
    print(f"  Source: {source_file}")

Document sources:

- health_psychology_course_syllabus
  Source: data/raw/course_syllabus.pdf
- michie_2011_behaviour_change_wheel
  Source: data/raw/michie_2011_behaviour_change_wheel.html
- ogden_2019_health_psychology
  Source: data/raw/ogden_2019_health_psychology.pdf
- wright_2019_3p_disease_model
  Source: data/raw/wright_2019_3p_disease_model.html


### Inspect the Homework 2 test queries

The previous retrieval output is displayed to identify the queries used in Homework 2.

These queries will be reused without changing their wording.

In [18]:
HW2_OUTPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "retrieval_examples.md"
)

if not HW2_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Homework 2 output was not found: "
        f"{HW2_OUTPUT_PATH}"
    )


hw2_output = HW2_OUTPUT_PATH.read_text(
    encoding="utf-8"
)

print(hw2_output)

# Semantic Retrieval Examples

**Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`

**Vector index:** `FAISS IndexFlatIP`

**Top-k:** `3`

## Query 1

**Query:** What is the biopsychosocial model of health?

### Top-1

- **Chunk ID:** `ogden_2019_health_psychology__0013`
- **Score:** `0.7959`
- **Document ID:** `ogden_2019_health_psychology`
- **Source:** `data/raw/ogden_2019_health_psychology.pdf`
- **Section:** `1.The Biopsychosocial Model`
- **Text preview:** smoking), pressures to change behavior (e.g. peer group expectations, parental pressure), social values on health (e.g. whether health was regarded as a good or a bad thing), social class, the environment, and ethnicity. #### Fig 1 The biopsychosocial model of health and illness (after Engel 1977, 1980) ![Fig 1 The biopsychosocial model of health a

### Top-2

- **Chunk ID:** `wright_2019_3p_disease_model__0010`
- **Score:** `0.7779`
- **Document ID:** `wright_2019_3p_disease_model`
- **Source:** `data/raw/wright_2019

### Stage 2 result

The retrieval chunks, FAISS index, document collection, source files, and Homework 2 retrieval output were inspected successfully.

The dataset contains 474 chunks across four documents. The FAISS index contains the same number of vectors as the chunk collection.

The exact six Homework 2 queries are now available for the evaluation set.

## Stage 3 — Define the evaluation set

Prepare the test cases used to compare the baseline and improved retrieval pipelines.

The same six queries from Homework 2 are reused without changing their wording.

Each test case contains:

- the query;
- the expected document;
- a metadata filter for the improved pipeline.

The expected document is used only for evaluation and is not passed to the retrieval algorithm.

Before defining the test cases, each document is assigned a document type that can be used for metadata filtering.

### Add document-type metadata

The lesson demonstration filters retrieval results using the `document_type` metadata field.

The current dataset does not contain this field, so each document is assigned one of three document types:

- `syllabus`;
- `textbook`;
- `research_article`.

This metadata is added only to the chunks loaded in memory. The original JSONL file is not modified.

In [19]:
DOCUMENT_TYPE_BY_DOCUMENT = {
    "health_psychology_course_syllabus": "syllabus",
    "ogden_2019_health_psychology": "textbook",
    "michie_2011_behaviour_change_wheel": "research_article",
    "wright_2019_3p_disease_model": "research_article",
}


for chunk in chunks:
    document_id = chunk["document_id"]

    chunk["metadata"]["document_type"] = (
        DOCUMENT_TYPE_BY_DOCUMENT[document_id]
    )


print(
    json.dumps(
        chunks[0]["metadata"],
        indent=2,
        ensure_ascii=False,
    )
)

{
  "document_id": "health_psychology_course_syllabus",
  "source_file": "data/raw/course_syllabus.pdf",
  "section": "Document overview",
  "chunk_index": 0,
  "document_type": "syllabus"
}


In [21]:
TEST_QUERIES = [
    {
        "query": "What is the biopsychosocial model of health?",
        "expected_document": "ogden_2019_health_psychology",
        "metadata_filter": {
            "document_type": "textbook"
        },
    },
    {
        "query": "What are the components of the COM-B model?",
        "expected_document": "michie_2011_behaviour_change_wheel",
        "metadata_filter": {
            "document_type": "research_article"
        },
    },
    {
        "query": (
            "How does the Behaviour Change Wheel "
            "support intervention design?"
        ),
        "expected_document": "michie_2011_behaviour_change_wheel",
        "metadata_filter": {
            "document_type": "research_article"
        },
    },
    {
        "query": (
            "What are the predisposing, precipitating, "
            "and perpetuating factors in the 3P model?"
        ),
        "expected_document": "wright_2019_3p_disease_model",
        "metadata_filter": {
            "document_type": "research_article"
        },
    },
    {
        "query": "How can stress affect physical health?",
        "expected_document": "ogden_2019_health_psychology",
        "metadata_filter": {
            "document_type": "textbook"
        },
    },
    {
        "query": (
            "What topics are covered in the "
            "Health Psychology course?"
        ),
        "expected_document": "health_psychology_course_syllabus",
        "metadata_filter": {
            "document_type": "syllabus"
        },
    },
]

print(f"Test queries: {len(TEST_QUERIES)}")

Test queries: 6


In [22]:
for number, test_case in enumerate(
    TEST_QUERIES,
    start=1,
):
    print(f"Query {number}: {test_case['query']}")
    print(
        "Expected document:",
        test_case["expected_document"],
    )
    print(
        "Metadata filter:",
        test_case["metadata_filter"],
    )
    print("-" * 80)

Query 1: What is the biopsychosocial model of health?
Expected document: ogden_2019_health_psychology
Metadata filter: {'document_type': 'textbook'}
--------------------------------------------------------------------------------
Query 2: What are the components of the COM-B model?
Expected document: michie_2011_behaviour_change_wheel
Metadata filter: {'document_type': 'research_article'}
--------------------------------------------------------------------------------
Query 3: How does the Behaviour Change Wheel support intervention design?
Expected document: michie_2011_behaviour_change_wheel
Metadata filter: {'document_type': 'research_article'}
--------------------------------------------------------------------------------
Query 4: What are the predisposing, precipitating, and perpetuating factors in the 3P model?
Expected document: wright_2019_3p_disease_model
Metadata filter: {'document_type': 'research_article'}
-------------------------------------------------------------------

## Stage 4 — Run baseline retrieval diagnostics

In this stage, we reproduce the semantic retrieval pipeline from Homework 2.

For each query:

1. the query is converted into an embedding;
2. the embedding is normalized;
3. FAISS returns the top three semantic matches;
4. the retrieved document IDs are compared with the expected document.

No metadata filtering or hybrid scoring is applied at this stage.

In [23]:
from sentence_transformers import SentenceTransformer


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 3


model = SentenceTransformer(MODEL_NAME)

print("Embedding model:", MODEL_NAME)
print("Top-k:", TOP_K)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Top-k: 3


### Baseline semantic search

The baseline uses only vector similarity.

For each retrieved chunk, we store:

- semantic score;
- chunk ID;
- text;
- metadata.

In [24]:
def semantic_search(query, top_k=TOP_K):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
    )

    query_embedding = query_embedding.astype("float32")
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        top_k,
    )

    results = []

    for score, chunk_index in zip(
        scores[0],
        indices[0],
    ):
        if chunk_index == -1:
            continue

        chunk = chunks[chunk_index]

        results.append(
            {
                "score": float(score),
                "chunk_id": chunk["chunk_id"],
                "text": chunk["text"],
                "metadata": chunk.get("metadata", {}),
            }
        )

    return results

In [25]:
def print_diagnostic_hint(
    results,
    expected_document,
):
    retrieved_documents = [
        result["metadata"].get("document_id")
        for result in results
    ]

    if retrieved_documents[0] == expected_document:
        print(
            "Diagnostic hint: "
            "Top result matches the expected document."
        )

    elif expected_document in retrieved_documents:
        print(
            "Diagnostic hint: "
            "Expected document was retrieved, "
            "but not ranked first."
        )

    else:
        print(
            "Diagnostic hint: "
            "Expected document was not retrieved in top-k."
        )

In [26]:
baseline_results = []


for number, test_case in enumerate(
    TEST_QUERIES,
    start=1,
):
    query = test_case["query"]
    expected_document = test_case["expected_document"]

    results = semantic_search(
        query=query,
        top_k=TOP_K,
    )

    print("=" * 80)
    print(f"Query {number}: {query}")
    print("Expected document:", expected_document)
    print()

    for rank, result in enumerate(
        results,
        start=1,
    ):
        metadata = result["metadata"]

        print(f"Top-{rank}")
        print("Score:", round(result["score"], 4))
        print("Chunk ID:", result["chunk_id"])
        print(
            "Document ID:",
            metadata.get("document_id"),
        )
        print(
            "Source:",
            metadata.get("source_file"),
        )
        print(
            "Section:",
            metadata.get("section"),
        )
        print(
            "Text preview:",
            result["text"][:300],
        )
        print()

    print_diagnostic_hint(
        results=results,
        expected_document=expected_document,
    )

    baseline_results.append(
        {
            "query": query,
            "expected_document": expected_document,
            "results": results,
        }
    )

    print()

Query 1: What is the biopsychosocial model of health?
Expected document: ogden_2019_health_psychology

Top-1
Score: 0.7959
Chunk ID: ogden_2019_health_psychology__0013
Document ID: ogden_2019_health_psychology
Source: data/raw/ogden_2019_health_psychology.pdf
Section: 1.The Biopsychosocial Model
Text preview: smoking), pressures to change behavior (e.g. peer group expectations, parental pressure), social values on health (e.g. whether health was regarded as a good or a bad thing), social class, the environment, and ethnicity. #### Fig 1 The biopsychosocial model of health and illness (after Engel 1977, 1

Top-2
Score: 0.7779
Chunk ID: wright_2019_3p_disease_model__0010
Document ID: wright_2019_3p_disease_model
Source: data/raw/wright_2019_3p_disease_model.html
Section: Introduction
Text preview: and disease are conceptualized or managed today ([Kontos, 2011](https://pmc.ncbi.nlm.nih.gov/articles/PMC6879427/#B55)). This could be, in part, because the biopsychosocial model lacks a framew

In [27]:
baseline_top_1_correct = 0
baseline_recall_at_3 = 0


for evaluation in baseline_results:
    expected_document = evaluation["expected_document"]

    retrieved_documents = [
        result["metadata"].get("document_id")
        for result in evaluation["results"]
    ]

    if retrieved_documents[0] == expected_document:
        baseline_top_1_correct += 1

    if expected_document in retrieved_documents:
        baseline_recall_at_3 += 1


total_queries = len(baseline_results)

baseline_top_1_accuracy = (
    baseline_top_1_correct / total_queries
)

baseline_recall_at_3_score = (
    baseline_recall_at_3 / total_queries
)


print(
    "Baseline Top-1 accuracy:",
    f"{baseline_top_1_correct}/{total_queries}",
    f"= {baseline_top_1_accuracy:.1%}",
)

print(
    "Baseline Recall@3:",
    f"{baseline_recall_at_3}/{total_queries}",
    f"= {baseline_recall_at_3_score:.1%}",
)

Baseline Top-1 accuracy: 5/6 = 83.3%
Baseline Recall@3: 6/6 = 100.0%


## Stage 5 — Add metadata filtering

In this stage, we add exact-match filtering based on document metadata.

FAISS first retrieves a larger set of semantic candidates. The metadata filter then removes candidates whose document type does not match the filter defined for the query.

Metadata filtering does not change semantic scores. It only narrows the candidate set.

In [28]:
CANDIDATE_K = 10
FINAL_K = 3


print("Candidate-k:", CANDIDATE_K)
print("Final-k:", FINAL_K)

Candidate-k: 10
Final-k: 3


In [29]:
def metadata_matches(
    result,
    metadata_filter,
):
    metadata = result.get("metadata", {})

    for key, expected_value in metadata_filter.items():
        if metadata.get(key) != expected_value:
            return False

    return True

In [30]:
def apply_metadata_filter(
    results,
    metadata_filter,
    final_k=FINAL_K,
):
    filtered_results = [
        result
        for result in results
        if metadata_matches(
            result,
            metadata_filter,
        )
    ]

    return filtered_results[:final_k]

In [31]:
metadata_filter_results = []


for number, test_case in enumerate(
    TEST_QUERIES,
    start=1,
):
    query = test_case["query"]
    expected_document = test_case["expected_document"]
    metadata_filter = test_case["metadata_filter"]

    candidates = semantic_search(
        query=query,
        top_k=CANDIDATE_K,
    )

    filtered_results = apply_metadata_filter(
        results=candidates,
        metadata_filter=metadata_filter,
        final_k=FINAL_K,
    )

    print("=" * 80)
    print(f"Query {number}: {query}")
    print("Metadata filter:", metadata_filter)
    print("Candidates before filtering:", len(candidates))
    print("Results after filtering:", len(filtered_results))
    print()

    for rank, result in enumerate(
        filtered_results,
        start=1,
    ):
        metadata = result["metadata"]

        print(f"Top-{rank}")
        print("Score:", round(result["score"], 4))
        print("Chunk ID:", result["chunk_id"])
        print(
            "Document ID:",
            metadata.get("document_id"),
        )
        print(
            "Document type:",
            metadata.get("document_type"),
        )
        print(
            "Section:",
            metadata.get("section"),
        )
        print()

    print_diagnostic_hint(
        results=filtered_results,
        expected_document=expected_document,
    )

    metadata_filter_results.append(
        {
            "query": query,
            "expected_document": expected_document,
            "metadata_filter": metadata_filter,
            "candidates_before_filtering": len(candidates),
            "results_after_filtering": len(filtered_results),
            "results": filtered_results,
        }
    )

    print()

Query 1: What is the biopsychosocial model of health?
Metadata filter: {'document_type': 'textbook'}
Candidates before filtering: 10
Results after filtering: 3

Top-1
Score: 0.7959
Chunk ID: ogden_2019_health_psychology__0013
Document ID: ogden_2019_health_psychology
Document type: textbook
Section: 1.The Biopsychosocial Model

Top-2
Score: 0.7682
Chunk ID: ogden_2019_health_psychology__0012
Document ID: ogden_2019_health_psychology
Document type: textbook
Section: 1.The Biopsychosocial Model

Top-3
Score: 0.7248
Chunk ID: ogden_2019_health_psychology__0004
Document ID: ogden_2019_health_psychology
Document type: textbook
Section: Overview

Diagnostic hint: Top result matches the expected document.

Query 2: What are the components of the COM-B model?
Metadata filter: {'document_type': 'research_article'}
Candidates before filtering: 10
Results after filtering: 3

Top-1
Score: 0.4078
Chunk ID: michie_2011_behaviour_change_wheel__0032
Document ID: michie_2011_behaviour_change_wheel
Docu

## Stage 6 — Add simple hybrid retrieval

In this stage, we combine semantic similarity with a simple keyword-overlap score.

The improved pipeline performs:

1. semantic retrieval of ten candidates;
2. metadata filtering;
3. keyword-overlap scoring;
4. hybrid reranking;
5. selection of the final top three results.

This is a lightweight teaching implementation and does not use BM25 or a production reranker.

In [32]:
import re


SEMANTIC_WEIGHT = 0.7
KEYWORD_WEIGHT = 0.3


print("Semantic weight:", SEMANTIC_WEIGHT)
print("Keyword weight:", KEYWORD_WEIGHT)

Semantic weight: 0.7
Keyword weight: 0.3


### Keyword-overlap score

The query and chunk text are converted into lowercase word sets.

The keyword score represents the proportion of unique query terms that also appear in the chunk text.

In [33]:
def tokenize(text):
    return set(
        re.findall(
            r"\b\w+\b",
            text.lower(),
        )
    )


def keyword_overlap_score(query, text):
    query_terms = tokenize(query)
    text_terms = tokenize(text)

    if not query_terms:
        return 0.0

    shared_terms = query_terms.intersection(
        text_terms
    )

    return len(shared_terms) / len(query_terms)

### Hybrid score

The final hybrid score combines:

- 70% semantic similarity;
- 30% keyword overlap.

Candidates are sorted by the hybrid score in descending order.

In [34]:
def add_hybrid_scores(query, results):
    hybrid_results = []

    for result in results:
        keyword_score = keyword_overlap_score(
            query=query,
            text=result["text"],
        )

        hybrid_score = (
            SEMANTIC_WEIGHT * result["score"]
            + KEYWORD_WEIGHT * keyword_score
        )

        hybrid_result = result.copy()
        hybrid_result["keyword_score"] = keyword_score
        hybrid_result["hybrid_score"] = hybrid_score

        hybrid_results.append(hybrid_result)

    hybrid_results.sort(
        key=lambda result: result["hybrid_score"],
        reverse=True,
    )

    return hybrid_results

### Run the improved retrieval pipeline

For every query, FAISS returns ten semantic candidates.

The metadata filter is applied first. The remaining candidates are then reranked using the combined semantic and keyword scores.

The final top three results are retained.

In [35]:
improved_results = []


for number, test_case in enumerate(
    TEST_QUERIES,
    start=1,
):
    query = test_case["query"]
    expected_document = test_case["expected_document"]
    metadata_filter = test_case["metadata_filter"]

    candidates = semantic_search(
        query=query,
        top_k=CANDIDATE_K,
    )

    filtered_candidates = apply_metadata_filter(
        results=candidates,
        metadata_filter=metadata_filter,
        final_k=CANDIDATE_K,
    )

    ranked_results = add_hybrid_scores(
        query=query,
        results=filtered_candidates,
    )

    final_results = ranked_results[:FINAL_K]

    print("=" * 80)
    print(f"Query {number}: {query}")
    print("Metadata filter:", metadata_filter)
    print("Semantic candidates:", len(candidates))
    print(
        "Candidates after filtering:",
        len(filtered_candidates),
    )
    print()

    for rank, result in enumerate(
        final_results,
        start=1,
    ):
        metadata = result["metadata"]

        print(f"Top-{rank}")
        print(
            "Semantic score:",
            round(result["score"], 4),
        )
        print(
            "Keyword score:",
            round(result["keyword_score"], 4),
        )
        print(
            "Hybrid score:",
            round(result["hybrid_score"], 4),
        )
        print("Chunk ID:", result["chunk_id"])
        print(
            "Document ID:",
            metadata.get("document_id"),
        )
        print(
            "Document type:",
            metadata.get("document_type"),
        )
        print(
            "Section:",
            metadata.get("section"),
        )
        print()

    print_diagnostic_hint(
        results=final_results,
        expected_document=expected_document,
    )

    improved_results.append(
        {
            "query": query,
            "expected_document": expected_document,
            "metadata_filter": metadata_filter,
            "results": final_results,
        }
    )

    print()

Query 1: What is the biopsychosocial model of health?
Metadata filter: {'document_type': 'textbook'}
Semantic candidates: 10
Candidates after filtering: 3

Top-1
Semantic score: 0.7959
Keyword score: 0.7143
Hybrid score: 0.7715
Chunk ID: ogden_2019_health_psychology__0013
Document ID: ogden_2019_health_psychology
Document type: textbook
Section: 1.The Biopsychosocial Model

Top-2
Semantic score: 0.7248
Keyword score: 0.8571
Hybrid score: 0.7645
Chunk ID: ogden_2019_health_psychology__0004
Document ID: ogden_2019_health_psychology
Document type: textbook
Section: Overview

Top-3
Semantic score: 0.7682
Keyword score: 0.7143
Hybrid score: 0.752
Chunk ID: ogden_2019_health_psychology__0012
Document ID: ogden_2019_health_psychology
Document type: textbook
Section: 1.The Biopsychosocial Model

Diagnostic hint: Top result matches the expected document.

Query 2: What are the components of the COM-B model?
Metadata filter: {'document_type': 'research_article'}
Semantic candidates: 10
Candidate

In [36]:
improved_top_1_correct = 0
improved_recall_at_3 = 0


for evaluation in improved_results:
    expected_document = evaluation["expected_document"]

    retrieved_documents = [
        result["metadata"].get("document_id")
        for result in evaluation["results"]
    ]

    if (
        retrieved_documents
        and retrieved_documents[0] == expected_document
    ):
        improved_top_1_correct += 1

    if expected_document in retrieved_documents:
        improved_recall_at_3 += 1


total_queries = len(improved_results)

improved_top_1_accuracy = (
    improved_top_1_correct / total_queries
)

improved_recall_at_3_score = (
    improved_recall_at_3 / total_queries
)


print(
    "Improved Top-1 accuracy:",
    f"{improved_top_1_correct}/{total_queries}",
    f"= {improved_top_1_accuracy:.1%}",
)

print(
    "Improved Recall@3:",
    f"{improved_recall_at_3}/{total_queries}",
    f"= {improved_recall_at_3_score:.1%}",
)

Improved Top-1 accuracy: 6/6 = 100.0%
Improved Recall@3: 6/6 = 100.0%


### Stage 6 result

The metadata-filtered candidates were reranked using a weighted combination of semantic similarity and keyword overlap.

The improved pipeline achieved:

- Top-1 accuracy: 100%;
- Recall@3: 100%.

Hybrid scoring changed the internal chunk ranking for several queries but did not improve document-level accuracy beyond metadata filtering.

For the 3P-model query, the hybrid method promoted a chunk with complete keyword overlap above the more direct baseline explanation. This shows that simple keyword overlap can sometimes overvalue lexical matches.

Therefore, metadata filtering produced the largest positive effect in this experiment, while hybrid reranking had a mixed effect on chunk-level relevance.

## Stage 7 — Compare baseline and improved retrieval

In this stage, we compare the baseline semantic retrieval results with the final improved pipeline.

For each query, we compare:

- the expected document;
- the baseline top result;
- the improved top result;
- whether the document-level result improved;
- whether hybrid scoring changed the top-ranked chunk.

This comparison provides evidence of how the retrieval pipeline changed after metadata filtering and hybrid reranking.

In [37]:
comparison_results = []


for baseline_evaluation, improved_evaluation in zip(
    baseline_results,
    improved_results,
):
    query = baseline_evaluation["query"]
    expected_document = baseline_evaluation["expected_document"]

    baseline_top_1 = baseline_evaluation["results"][0]
    improved_top_1 = improved_evaluation["results"][0]

    baseline_document = (
        baseline_top_1["metadata"].get("document_id")
    )
    improved_document = (
        improved_top_1["metadata"].get("document_id")
    )

    baseline_chunk = baseline_top_1["chunk_id"]
    improved_chunk = improved_top_1["chunk_id"]

    baseline_correct = (
        baseline_document == expected_document
    )
    improved_correct = (
        improved_document == expected_document
    )

    if not baseline_correct and improved_correct:
        change = (
            "Improved: the expected document moved "
            "to rank one after metadata filtering."
        )

    elif baseline_correct and not improved_correct:
        change = (
            "Degraded: the improved pipeline no longer "
            "returns the expected document at rank one."
        )

    elif baseline_chunk != improved_chunk:
        change = (
            "The expected document remained correct, "
            "but hybrid scoring changed the top chunk."
        )

    else:
        change = (
            "No change: the correct top result "
            "was preserved."
        )

    comparison_results.append(
        {
            "query": query,
            "expected_document": expected_document,
            "baseline_top_1_chunk": baseline_chunk,
            "baseline_top_1_document": baseline_document,
            "improved_top_1_chunk": improved_chunk,
            "improved_top_1_document": improved_document,
            "baseline_correct": baseline_correct,
            "improved_correct": improved_correct,
            "change": change,
        }
    )


print(
    f"Comparison records: {len(comparison_results)}"
)

Comparison records: 6


In [38]:
for number, result in enumerate(
    comparison_results,
    start=1,
):
    print("=" * 80)
    print(f"Query {number}: {result['query']}")
    print(
        "Expected document:",
        result["expected_document"],
    )
    print(
        "Baseline top-1:",
        result["baseline_top_1_chunk"],
    )
    print(
        "Baseline document:",
        result["baseline_top_1_document"],
    )
    print(
        "Improved top-1:",
        result["improved_top_1_chunk"],
    )
    print(
        "Improved document:",
        result["improved_top_1_document"],
    )
    print("Change:", result["change"])
    print()

Query 1: What is the biopsychosocial model of health?
Expected document: ogden_2019_health_psychology
Baseline top-1: ogden_2019_health_psychology__0013
Baseline document: ogden_2019_health_psychology
Improved top-1: ogden_2019_health_psychology__0013
Improved document: ogden_2019_health_psychology
Change: No change: the correct top result was preserved.

Query 2: What are the components of the COM-B model?
Expected document: michie_2011_behaviour_change_wheel
Baseline top-1: michie_2011_behaviour_change_wheel__0032
Baseline document: michie_2011_behaviour_change_wheel
Improved top-1: michie_2011_behaviour_change_wheel__0032
Improved document: michie_2011_behaviour_change_wheel
Change: No change: the correct top result was preserved.

Query 3: How does the Behaviour Change Wheel support intervention design?
Expected document: michie_2011_behaviour_change_wheel
Baseline top-1: michie_2011_behaviour_change_wheel__0059
Baseline document: michie_2011_behaviour_change_wheel
Improved top-1: 

In [39]:
improved_queries = sum(
    1
    for result in comparison_results
    if (
        not result["baseline_correct"]
        and result["improved_correct"]
    )
)

unchanged_correct_queries = sum(
    1
    for result in comparison_results
    if (
        result["baseline_correct"]
        and result["improved_correct"]
    )
)

degraded_queries = sum(
    1
    for result in comparison_results
    if (
        result["baseline_correct"]
        and not result["improved_correct"]
    )
)

top_chunk_changes = sum(
    1
    for result in comparison_results
    if (
        result["baseline_top_1_chunk"]
        != result["improved_top_1_chunk"]
    )
)


print("Baseline Top-1 accuracy:", f"{baseline_top_1_accuracy:.1%}")
print("Improved Top-1 accuracy:", f"{improved_top_1_accuracy:.1%}")
print("Queries improved:", improved_queries)
print(
    "Queries remaining correct:",
    unchanged_correct_queries,
)
print("Queries degraded:", degraded_queries)
print("Top-1 chunk changes:", top_chunk_changes)

Baseline Top-1 accuracy: 83.3%
Improved Top-1 accuracy: 100.0%
Queries improved: 1
Queries remaining correct: 5
Queries degraded: 0
Top-1 chunk changes: 2


### Stage 7 result

The improved pipeline increased document-level Top-1 accuracy from 83.3% to 100%.

Metadata filtering produced the main positive effect by moving the course syllabus from rank two to rank one for the course-topics query.

Hybrid scoring changed the ranking of individual chunks for some queries. These changes did not further improve document-level accuracy and were not always clearly better in terms of answer relevance.

No query was degraded at the document level.

## Stage 8 — Generate the comparison report

In this stage, we export the baseline and improved retrieval results to the Markdown report required for submission.

The report contains:

- the query-level comparison;
- baseline and improved accuracy;
- a short analysis;
- the final conclusion.

In [40]:
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

COMPARISON_PATH = (
    OUTPUTS_DIR
    / "retrieval_comparison.md"
)


report_lines = [
    "# Retrieval Comparison",
    "",
    "## Evaluation Summary",
    "",
    f"- Baseline Top-1 accuracy: "
    f"{baseline_top_1_accuracy:.1%}",
    f"- Improved Top-1 accuracy: "
    f"{improved_top_1_accuracy:.1%}",
    f"- Baseline Recall@3: "
    f"{baseline_recall_at_3_score:.1%}",
    f"- Improved Recall@3: "
    f"{improved_recall_at_3_score:.1%}",
    f"- Queries improved: {improved_queries}",
    f"- Queries degraded: {degraded_queries}",
    "",
    "## Query-Level Comparison",
    "",
    "| Query | Baseline top-1 | Improved top-1 | What changed |",
    "|---|---|---|---|",
]


for result in comparison_results:
    row = (
        f"| {result['query']} "
        f"| `{result['baseline_top_1_chunk']}` "
        f"| `{result['improved_top_1_chunk']}` "
        f"| {result['change']} |"
    )

    report_lines.append(row)


report_lines.extend(
    [
        "",
        "## Analysis",
        "",
        (
            "Metadata filtering produced the largest positive effect. "
            "For the course-topics query, it removed textbook chunks "
            "and moved the syllabus from rank two to rank one."
        ),
        "",
        (
            "Hybrid scoring changed the chunk ranking for the 3P-model "
            "query. The promoted chunk had stronger keyword overlap, "
            "but it was not necessarily a more direct answer than the "
            "baseline top chunk."
        ),
        "",
        "## Conclusion",
        "",
        (
            "The improved pipeline increased document-level Top-1 "
            "accuracy from 83.3% to 100%. Metadata filtering was the "
            "most effective improvement, while simple hybrid reranking "
            "had a mixed effect on chunk-level relevance."
        ),
    ]
)


COMPARISON_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)


print("✅ Comparison report created:")
print(COMPARISON_PATH)

✅ Comparison report created:
/content/health-psychology-rag-kb/outputs/retrieval_comparison.md


In [41]:
print(
    COMPARISON_PATH.read_text(
        encoding="utf-8"
    )
)

# Retrieval Comparison

## Evaluation Summary

- Baseline Top-1 accuracy: 83.3%
- Improved Top-1 accuracy: 100.0%
- Baseline Recall@3: 100.0%
- Improved Recall@3: 100.0%
- Queries improved: 1
- Queries degraded: 0

## Query-Level Comparison

| Query | Baseline top-1 | Improved top-1 | What changed |
|---|---|---|---|
| What is the biopsychosocial model of health? | `ogden_2019_health_psychology__0013` | `ogden_2019_health_psychology__0013` | No change: the correct top result was preserved. |
| What are the components of the COM-B model? | `michie_2011_behaviour_change_wheel__0032` | `michie_2011_behaviour_change_wheel__0032` | No change: the correct top result was preserved. |
| How does the Behaviour Change Wheel support intervention design? | `michie_2011_behaviour_change_wheel__0059` | `michie_2011_behaviour_change_wheel__0059` | No change: the correct top result was preserved. |
| What are the predisposing, precipitating, and perpetuating factors in the 3P model? | `wright_2019_3p_